Simulate summons from Fire Emblem Heroes.

1. Simulate drawing from pools with given weights (5\*, 4\*S, 4\*, 3\*). 
2. Get counts of units in summon pools from wiki.
3. Add colour to random summons.
4. Draw 5 at a time.
5. Add orb cost to summon.
6. Summon according to preference rules, somehow. (sorting?)
7. Get orb cost until certain target reached.
8. Simulate a lot and then average it.

In [6]:
import random
import string

from enum import Enum, auto
from dataclasses import dataclass
from collections import defaultdict

random.seed("testing")

In [7]:
class PoolName(Enum):
    five_focus = auto()
    five = auto()
    four_focus = auto()
    four_special_sr = auto()  # special hero special rate
    four_sr = auto()
    four = auto()
    three = auto()


class Rarity(Enum):
    one_star = auto()
    two_star = auto()
    three_star = auto()
    four_star = auto()
    five_star = auto()


class Colour(Enum):
    red = auto()
    blue = auto()
    green = auto()
    colourless = auto()


@dataclass(frozen=True)
class Unit:
    __slots__ = "unit_id", "name", "colour", "rarity"
    unit_id: int
    name: str
    colour: Colour
    rarity: Rarity
        

unit_dict = defaultdict(lambda: "".join(random.choices(string.ascii_letters, k=10)))

colours = tuple(Colour)
colour_dict = defaultdict(lambda: random.choice(colours))

In [8]:
@dataclass(frozen=True)
class Pool:
    __slots__ = "name", "units"
    name: PoolName
    units: tuple
        
    def get_unit(self):
        return random.choice(self.units)

In [9]:
class Banner:
    __slots__ = "pools", "focus_charges"
    
    def __init__(self, focus_charges=True):
        self.pools = {}
        self.focus_charges = focus_charges
        
    def add_pool(self, pool, weight):
        self.pools[pool] = weight
        
    def set_weight(self, pool, weight):
        self.pools[pool] = weight
        
    def draw(self, k=1):
        chosen_pools = random.choices(tuple(self.pools.keys()), tuple(self.pools.values()), k=k)
        units = [pool.get_unit() for pool in chosen_pools]
        return units

In [10]:
# for pool_name in PoolName:
#     print(pool_name)
    
five_focus_ids = tuple(range(1, 5))
five_ids = tuple(range(5, 21))
four_focus_ids = (4,)
four_sr_ids = tuple(range(21, 31))
four_ids = tuple(range(31, 61))
three_ids = tuple(range(31, 61))

five_focus_units = tuple(Unit(i, unit_dict[i], colour_dict[i], Rarity.five_star) for i in five_focus_ids)
five_units = tuple(Unit(i, unit_dict[i], colour_dict[i], Rarity.five_star) for i in five_ids)
four_focus_units = tuple(Unit(i, unit_dict[i], colour_dict[i], Rarity.five_star) for i in four_focus_ids)
four_units = tuple(Unit(i, unit_dict[i], colour_dict[i], Rarity.four_star) for i in four_ids)
three_units = tuple(Unit(i, unit_dict[i], colour_dict[i], Rarity.three_star) for i in three_ids)

pools = {
    Pool(PoolName.five_focus, five_focus_units): 3,
    Pool(PoolName.five, five_units): 3,
    Pool(PoolName.four_focus, four_focus_units): 3,
    Pool(PoolName.four, four_units): 52,
    Pool(PoolName.three, three_units): 36
}

# five_focus_pool = Pool(PoolName.five_focus, five_focus_units)

# print(five_focus_pool)
# print(five_focus_pool.get_unit())

banner = Banner()
for pool, weight in pools.items():
    banner.add_pool(pool, weight)
    
for unit in banner.draw(k=200):
    if unit.rarity == Rarity.five_star:
        print(unit)

Unit(unit_id=4, name='BJmpVMhoKu', colour=<Colour.blue: 2>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=11, name='TCzakcfTrF', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=1, name='uBDauWTfnE', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=1, name='uBDauWTfnE', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=1, name='uBDauWTfnE', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=12, name='RKKoFLwgFx', colour=<Colour.blue: 2>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=18, name='HozLfScGBi', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=14, name='AVyzpNLfZs', colour=<Colour.red: 1>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=4, name='BJmpVMhoKu', colour=<Colour.blue: 2>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=17, name='fBhAMnAAiv', colour=<Colour.green: 3>, rarity=<Rarity.five_star: 5>)
Unit(unit_id=4, name='BJmpVMhoKu', colour=<Colour.blue: 2>, rarity=<Rarity.five_star: 5>)
Unit(unit_

I want to get actual data, to use for the summoning simulation. This will involve the CargoQuery page on feheroes.fandom.com, i.e. the special page [here](https://feheroes.fandom.com/wiki/Special:CargoQuery).

I will add the relevant queries as markdown after making them.

To simplify the process, I will break it down into steps:
1. Get a list of all banners, in reverse start date order. This should include some sort of unique name or ID, so that I can use them later without worrying about name clashes. I will need to check what happens for banners that repeat each year.
2. Get a list of current banners. Possibly, this will just be a filter on the result from step 1.
3. For a specific normal heroes banner, get a list of the focus heroes and their possible rarities and colours. I will probably use "New Heroes: Fortune's Weave", since that is a currently ongoing normal heroes banner as of writing this.
4. For the same banner, get a list of the pools that it can draw from.
5. For each pool, get a list of all heroes and their rarities that the pool can give.

## Step 1: Get list of all banners

This can use the "SummoningEvents" cargo table. See an example query of this table [here](https://feheroes.fandom.com/wiki/Special:CargoQuery?tables=SummoningEvents&fields=_pageName%3DPage%2C+WikiName%3DWikiName%2C+Name%3DName%2C+StartTime%3DStartTime%2C+EndTime%3DEndTime%2C+EventType%3DEventType%2C+SummoningEvents._ID%2C+&where=&join_on=&group_by=&having=&order_by%5B0%5D=SummoningEvents.StartTime&order_by_options%5B0%5D=DESC&limit=200&offset=0&format=).

Example call: https://feheroes.fandom.com/wiki/Special:CargoExport?tables=SummoningEvents&&fields=_pageName%3DPage%2C+WikiName%3DWikiName%2C+Name%3DName%2C+StartTime%3DStartTime%2C+EndTime%3DEndTime%2C+EventType%3DEventType%2C+SummoningEvents._ID%2C+SummoningEvents._pageName%2C&&order+by=SummoningEvents.StartTime+DESC&limit=1500&offset=0&format=csv&filename=summoning_events.csv&parse+values=yes

It seems as though there are too many banners in FEH's history to request a CSV file of all of them at once. Either I use pagination to go through them all, or I only take a subset.

For now, I will simply take the last 30 banners.

Link: https://feheroes.fandom.com/wiki/Special:CargoExport?tables=SummoningEvents&&fields=_pageName%3DPage%2C+WikiName%3DWikiName%2C+Name%3DName%2C+StartTime%3DStartTime%2C+EndTime%3DEndTime%2C+EventType%3DEventType%2C+SummoningEvents._ID%2C+SummoningEvents._pageName%2C&&order+by=SummoningEvents.StartTime+DESC&limit=30&offset=0&format=csv&filename=summoning_events.csv

I have saved this as "summoning_events.csv". Note that some characters have been encoded in the CSV file, e.g. the `Name` field on the web view shows `Legendary & Mythic Hero Remix 1`, while the CSV file shows `Legendary &amp; Mythic Hero Remix 1`. The `WikiName` field seems to show a consistent name for each banner that could be useful to join results together when searching for details about a specific banner later.

## Step 2: Get list of currently running banners

This is based on the same query as above, but filtering by the current date being between `StartTime` and `EndTime` of the same SummoningEvents table.

Link: https://feheroes.fandom.com/wiki/Special:CargoExport?tables=SummoningEvents&&fields=_pageName%3DPage%2C+WikiName%3DWikiName%2C+Name%3DName%2C+StartTime%3DStartTime%2C+EndTime%3DEndTime%2C+EventType%3DEventType%2C+SummoningEvents._ID%2C+SummoningEvents._pageName%2C&where=StartTime+%3C%3D+NOW%28%29+AND+NOW%28%29+%3C+EndTime&order+by=SummoningEvents.StartTime+DESC&limit=100&offset=0&format=csv&filename=summoning_events_current.csv

## Step 3: Get details for a specific banner

For the specific banner, I will use the banner "New Heroes: Fortune's Weave" (WikiName: `New Heroes Fortunes Weave 20260908`). Note that this banner does not have any 4* focus heroes, so maybe a different banner with a 4* focus rate would be a better example.

Looking through the available tables, the one that looks useful for getting details on focus heroes is `SummoningEventFocuses`.

Example link: https://feheroes.fandom.com/wiki/Special:CargoQuery?tables=SummoningEventFocuses%2C+Units&fields=SummoningEventFocuses._pageName%3DPage%2CSummoningEventFocuses.WikiName%3DWikiName%2CSummoningEventFocuses.Unit%3DUnit%2CSummoningEventFocuses.Rarity%3DRarity%2CUnits.Name%3DUName%2CUnits.Title%3DUTitle%2C&where=SummoningEventFocuses.WikiName%3D%27New+Heroes+Fortunes+Weave+20260908%27&join_on=SummoningEventFocuses.Unit+%3D+Units.WikiName&group_by=&having=&order_by%5B0%5D=&order_by_options%5B0%5D=ASC&limit=100&offset=&format=

The Units table holds more info on Units, and can be joined using `SummoningEventFocuses.Unit=Units.WikiName`. However, Units table does not seem to have any info on the colour of the unit, which I need for summoning simulation. Instead, I can use the Units.WeaponType field and join it to WeaponTypes.WikiName, and then use WeaponTypes.Color to get the colour as a string.

Link: https://feheroes.fandom.com/wiki/Special:CargoExport?tables=SummoningEventFocuses%2C+Units%2C+WeaponTypes&join+on=SummoningEventFocuses.Unit%3DUnits.WikiName%2C+Units.WeaponType%3DWeaponTypes.WikiName&fields=SummoningEventFocuses._pageName%3DPage%2CSummoningEventFocuses.WikiName%3DWikiName%2CSummoningEventFocuses.Unit%3DUnit%2CSummoningEventFocuses.Rarity%3DRarity%2CUnits.IntID%2CUnits.Name%3DUName%2CUnits.Title%3DUTitle%2CUnits.WeaponType%3DWeaponType%2C+WeaponTypes.Color%3DColour%2C&where=SummoningEventFocuses.WikiName%3D%27New+Heroes+Fortunes+Weave+20260908%27&order+by=&limit=100&format=csv&filename=focus_units_fortunes_weave.csv

## Step 4: Get pools that a banner can pull from

This is where it might get tricky. I'm not sure if this data is anywhere in the Cargo tables, but it seems very relevant to all banners so I have a feeling it should be. I might need to look at properties or some sort of "banner type" if it's anywhere, if banners generally use some sort of template of what kind of banner they are, rather than listing the pools and their odds for every banner individually.